In [6]:
import os
import subprocess
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

# Change this to your project
PROJECT_PATH = "/Users/mukulsharma/testing/matter"

client = Groq(api_key=os.environ["GROQ_API_KEY"])


def execute_command(command: str):
    print(f"\n>>> {command}")

    result = subprocess.run(
        command,
        shell=True,
        cwd=PROJECT_PATH,
        capture_output=True,
        text=True,
    )

    print(result.stdout)
    if result.stderr:
        print(result.stderr)

    return result.returncode


system_prompt = """
You are a Git automation assistant.

Return ONLY a newline-separated list of git shell commands.

Do not explain anything.
Do not use markdown.
Do not use code fences.

Example:

git init
git add .
git commit -m "Initial commit"
git branch -M main
"""


task = input("What should I do?\n> ")

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    temperature=0,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": task},
    ],
)

commands_text = response.choices[0].message.content.strip()

print("\nAI generated commands:\n")
print(commands_text)

commands = [
    line.strip()
    for line in commands_text.splitlines()
    if line.strip()
]

confirm = input("\nExecute these commands? (y/n): ")

if confirm.lower() == "y":
    for cmd in commands:
        rc = execute_command(cmd)
        if rc != 0:
            print("Stopping because a command failed.")
            break


AI generated commands:

git init
git branch -M main
git checkout -b developing


: 